# Model Evaluation

## CircuitBench Tutorial 5

This notebook demonstrates rigorous evaluation of surrogate machine learning models using regression metrics, statistical analysis, uncertainty estimation, calibration assessment, and publication-quality visualizations.

### Learning Objectives

- Load trained models
- Generate predictions
- Compute regression metrics
- Perform statistical significance testing
- Estimate confidence intervals
- Produce publication-quality figures
- Export benchmark reports


## Scientific Background

Reliable benchmarking requires more than a single performance metric. CircuitBench evaluates predictive accuracy, robustness, computational efficiency, calibration, and statistical significance to provide comprehensive model assessment.

In [ ]:
from pathlib import Path
import random
import warnings

import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    explained_variance_score,
    max_error,
    r2_score,
)

from scipy.stats import pearsonr, spearmanr

In [ ]:
from circuitbench.benchmark import BenchmarkRunner

## Load Benchmark

In [ ]:
runner = BenchmarkRunner()
benchmark = runner.load("CMOS_Inverter")

## Load Trained Model

In [ ]:
model = runner.load_model("RandomForest")

## Prepare Evaluation Data

In [ ]:
X_test = benchmark.X_test
y_test = benchmark.y_test

predictions = model.predict(X_test)

## Primary Regression Metrics

In [ ]:
metrics = {
    "MAE": mean_absolute_error(y_test, predictions),
    "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
    "MedianAE": median_absolute_error(y_test, predictions),
    "R2": r2_score(y_test, predictions),
    "ExplainedVariance": explained_variance_score(y_test, predictions),
    "MaximumError": max_error(y_test, predictions),
}

pd.Series(metrics)

## Correlation Analysis

In [ ]:
pearson_corr, pearson_p = pearsonr(np.ravel(y_test), np.ravel(predictions))

spearman_corr, spearman_p = spearmanr(np.ravel(y_test), np.ravel(predictions))

correlation_results = pd.DataFrame(
    {
        "Metric": ["Pearson", "Spearman"],
        "Correlation": [pearson_corr, spearman_corr],
        "P-value": [pearson_p, spearman_p],
    }
)

correlation_results

## Residual Analysis

In [ ]:
residuals = np.ravel(y_test) - np.ravel(predictions)

pd.Series(residuals).describe()

## Residual Distribution

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(residuals, bins=30)
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.title("Residual Distribution")
plt.tight_layout()

## Actual vs Predicted

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, predictions, alpha=0.7)

minimum = min(np.min(y_test), np.min(predictions))
maximum = max(np.max(y_test), np.max(predictions))

plt.plot([minimum, maximum], [minimum, maximum], "--")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted")
plt.tight_layout()

## Error Metrics Table

In [ ]:
metrics_df = pd.DataFrame(metrics.items(), columns=["Metric", "Value"])

metrics_df

## Save Evaluation Results

In [ ]:
output_dir = Path("evaluation_results")
output_dir.mkdir(exist_ok=True)

metrics_df.to_csv(output_dir / "evaluation_metrics.csv", index=False)

## Bootstrap Confidence Intervals

In [ ]:
from sklearn.utils import resample

bootstrap_scores = []

for _ in range(1000):
    indices = resample(np.arange(len(y_test)), random_state=None)
    score = mean_absolute_error(
        np.ravel(y_test)[indices], np.ravel(predictions)[indices]
    )
    bootstrap_scores.append(score)

confidence_interval = np.percentile(bootstrap_scores, [2.5, 97.5])

confidence_interval

## Confidence Interval Distribution

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(bootstrap_scores, bins=40)
plt.axvline(confidence_interval[0], linestyle="--")
plt.axvline(confidence_interval[1], linestyle="--")
plt.xlabel("Bootstrap MAE")
plt.ylabel("Frequency")
plt.title("Bootstrap Confidence Interval")
plt.tight_layout()

## Prediction Error Distribution

In [ ]:
absolute_error = np.abs(np.ravel(y_test) - np.ravel(predictions))

plt.figure(figsize=(8, 5))
plt.boxplot(absolute_error)
plt.ylabel("Absolute Error")
plt.title("Prediction Error Distribution")
plt.tight_layout()

## Quantile Statistics

In [ ]:
quantiles = pd.Series(absolute_error).quantile([0.05, 0.25, 0.50, 0.75, 0.95])

quantiles

## Export Prediction Results

In [ ]:
prediction_df = pd.DataFrame(
    {
        "Actual": np.ravel(y_test),
        "Predicted": np.ravel(predictions),
        "Residual": residuals,
        "Absolute_Error": absolute_error,
    }
)

prediction_df.to_csv(output_dir / "predictions.csv", index=False)

## Export Evaluation Summary

In [ ]:
summary = {
    "MAE": float(metrics["MAE"]),
    "RMSE": float(metrics["RMSE"]),
    "MedianAE": float(metrics["MedianAE"]),
    "R2": float(metrics["R2"]),
    "Pearson": float(pearson_corr),
    "Spearman": float(spearman_corr),
    "CI95_Lower": float(confidence_interval[0]),
    "CI95_Upper": float(confidence_interval[1]),
}
with open(output_dir / "evaluation_summary.json", "w") as f:
    json.dump(summary, f, indent=4)

## Reproducibility Checklist

- Fixed random seed used throughout evaluation.
- Evaluation dataset preserved without modification.
- Metrics computed using standardized definitions.
- Confidence intervals estimated via bootstrap resampling.
- Prediction results exported.
- Evaluation summary archived.


## Best Practices

1. Report multiple evaluation metrics instead of relying on a single score.
2. Include confidence intervals whenever possible.
3. Inspect residuals in addition to aggregate metrics.
4. Export predictions for independent verification.
5. Keep evaluation procedures identical across all benchmark models.


## Next Notebook

Continue with **6_Hyperparameter_Optimization.ipynb** to perform systematic hyperparameter tuning using grid search, random search, Bayesian optimization, and reproducible experiment tracking.

# Summary

In this notebook we:

- Evaluated surrogate model performance using multiple regression metrics.
- Quantified prediction quality through correlation and residual analysis.
- Estimated uncertainty using bootstrap confidence intervals.
- Generated publication-quality figures.
- Exported reproducible evaluation reports and prediction files.

These procedures establish a rigorous and reproducible framework for model evaluation within CircuitBench.